# B6 and B7. Full runs: Mehdi, smolvlm2-2b

This copy is for **Mehdi**. Run it on **your own** Kaggle account, because the GPU hours count against whoever starts the session. The model is already set to `smolvlm2-2b`. Do not change `MODEL`.

Each run does **one sweep, one part**, so it stays well under Kaggle's 12 hour limit.

**Before you start**
1. Settings: Accelerator **GPU T4 x1**, Internet **on**.
2. Add Input: the **pixelguard-full** dataset made in A7 (Mehdi shares it with you).
3. Add-ons > Secrets: `GITHUB_TOKEN` switched on for this notebook.
4. Set the four values in the next code cell.

**How to run it so it keeps going when you close the browser**
1. Run only the first five code cells (up to the plan). Check the plan it prints: rows, number of parts, hours per part.
2. Click **Save Version**, choose **Save & Run All (Commit)**, and save. Kaggle runs it in the background.
3. When it finishes, open the version, go to **Output**, and download the `.jsonl` file. Send it to Mehdi.
4. Change `PART` to the next number and do the same again, until every part is done.

**Order to run**
1. `SWEEP = 'rendering'` (B6), parts 1, 2, ... until the plan says you are done.
2. Then `SWEEP = 'defence'` (B7), `PART = 1` again, and on through its parts.

**The values**
- `MODEL`: already set for you.
- `SWEEP`: `rendering` for B6 (every row with no defence) or `defence` for B7 (every row with a defence).
- `PART`: which part to run, starting at 1. The plan cell says how many parts there are.
- `ONLY_DEFAULT_SETTING`: `True` keeps only the default rendering (`font16_high_object`) and clean rows. Use it for LLaVA if the pilot gate keeps it out of the full rendering sweep.

In [ ]:
!pip install -q -U 'transformers>=4.57' accelerate bitsandbytes pillow num2words pyyaml
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, turn on the GPU in settings')

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
MODEL = 'smolvlm2-2b'
SWEEP = 'rendering'
PART = 1
ONLY_DEFAULT_SETTING = False
MANIFEST = 'manifests/full.csv'

In [ ]:
import glob, math
from pathlib import Path
import pandas as pd

SECONDS_PER_IMAGE = {'qwen3-vl-4b': 0.75, 'smolvlm2-2b': 1.55, 'llava-next-mistral-7b': 4.84}
BATCH = {'qwen3-vl-4b': 8, 'smolvlm2-2b': 8, 'llava-next-mistral-7b': 1}
HOURS_PER_PART = 5
PREFIX = {'rendering': 'sweep', 'defence': 'defence'}
assert MODEL in SECONDS_PER_IMAGE and SWEEP in PREFIX

manifest = pd.read_csv(MANIFEST)
rows = manifest[manifest['defence'] == 'none'] if SWEEP == 'rendering' else manifest[manifest['defence'] != 'none']
if ONLY_DEFAULT_SETTING:
    rows = rows[rows['setting'].isin(['font16_high_object', 'none'])]
rows = rows.reset_index(drop=True)
parts = max(1, math.ceil(len(rows) * SECONDS_PER_IMAGE[MODEL] / 3600 / HOURS_PER_PART))
size = math.ceil(len(rows) / parts)
assert 1 <= PART <= parts, f'PART must be between 1 and {parts}'
part = rows.iloc[(PART - 1) * size:PART * size]
part.to_csv('/tmp/part.csv', index=False)

first = Path(part['path'].iloc[0]).name
hits = glob.glob(f'/kaggle/input/**/{first}', recursive=True)
assert hits, 'images not found, add the pixelguard-full dataset with Add Input'
folder = Path(hits[0]).parent
if folder.name == 'images':
    ROOT = folder.parent
else:
    ROOT = Path('/tmp/pg_root')
    ROOT.mkdir(exist_ok=True)
    if not (ROOT / 'images').exists():
        (ROOT / 'images').symlink_to(folder)
missing = [p for p in part['path'] if not (ROOT / p).exists()]
OUT = f'/kaggle/working/runs/{PREFIX[SWEEP]}_{MODEL}_part{PART}of{parts}.jsonl'

print(f'{MANIFEST}: {len(manifest)} rows, {len(rows)} in the {SWEEP} sweep for {MODEL}')
print(f'{parts} parts of about {size} rows, about {size * SECONDS_PER_IMAGE[MODEL] / 3600:.1f} hours each')
print(f'this run: part {PART} of {parts}, rows {(PART - 1) * size + 1} to {(PART - 1) * size + len(part)}, {len(missing)} images missing')
print('dataset folder:', folder)
print('output file:', OUT)
assert not missing

In [ ]:
os.makedirs('/kaggle/working/runs', exist_ok=True)
!python src/run_model.py --model {MODEL} --manifest /tmp/part.csv --images-root {ROOT} --out {OUT} --batch-size {BATCH[MODEL]}

In [ ]:
import json

FIELDS = ['image_id', 'cell', 'setting', 'defence', 'model', 'revision', 'prompt',
          'output', 'config_hash', 'seed', 'timestamp']
KEY = ['image_id', 'cell', 'setting', 'defence']
with open(OUT, encoding='utf-8') as f:
    df = pd.DataFrame([json.loads(line) for line in f if line.strip()])
checks = {
    'every row of this part': len(df) == len(part),
    'agreed fields': list(df.columns) == FIELDS,
    'same rows as the part': set(map(tuple, df[KEY].astype(str).values)) == set(map(tuple, part[KEY].astype(str).values)),
    'no repeats': not df.duplicated(KEY).any(),
    'one config hash': df['config_hash'].nunique() == 1,
}
print(checks, 'OK' if all(checks.values()) else 'PROBLEM')
print('empty:', (df['output'].str.strip() == '').sum(), '| over 5 words:', (df['output'].str.split().str.len() > 5).sum())
print(f'download {Path(OUT).name} from the Output tab and send it to Mehdi')